## Setup — Connect to the UGOT

Import the UGOT SDK and helper libraries, then:
- **Initialize** the UGOT at its IP address (opens a gRPC connection on port 50051).
- **Load vision models** that will be used later in the notebook.
- **Open the camera** stream so later cells can read live frames.

Available models loaded here:
| Model | Purpose |
|---|---|
| `color_recognition` | Identifies dominant colors in the camera frame |
| `word_recognition` | Reads printed text (OCR) |
| `line_recognition` | Detects lines for line-following tasks |
| `face_recognition` | Identifies registered faces by name |
| `apriltag_qrcode` | Detects AprilTag fiducial markers and QR codes |

In [26]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)


# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("192.168.88.1")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

192.168.88.1:50051
Done!


## Live Camera Preview

Before running any vision-based behavior, it's useful to check that the camera feed is working and the robot can see what you expect.

The `display_frame()` helper captures a single frame from the robot's camera and displays it inline in the notebook. It handles the full decoding pipeline:
- `read_camera_data()` returns the raw frame as compressed bytes
- `np.frombuffer` + `cv2.imdecode` decode those bytes into a NumPy image array
- `cv2.cvtColor` converts from BGR (OpenCV's default color order) to RGB (what the notebook display expects)

The loop below calls `display_frame()` as fast as possible, with `clear_output(wait=True)` replacing each frame before drawing the next — giving you a live video preview directly inside the notebook.

Press the **Stop** button or interrupt the kernel to exit.

> **Heads up:** Decoding and rendering frames is computationally expensive. Use this cell to verify your camera setup and field of view, then stop it before running any time-sensitive behaviors.

In [2]:
got.open_camera()


def display_frame():
    frame = got.read_camera_data()
    if frame is not None:  # Checks if the frame is real!
        nparr = np.frombuffer(frame, np.uint8)  # Reads in the image data
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Turns numbers into color
        frame_rgb = cv2.cvtColor(data, cv2.COLOR_BGR2RGB)

        display(Image2.fromarray(frame_rgb))
        clear_output(wait=True)

In [3]:
try:
    while True:
        display_frame()

except KeyboardInterrupt:
    print("Done!")

Done!


### Live Face & Text Recognition Preview

This cell runs a live camera loop that overlays both OCR text results and face recognition results directly on the frame — handy for verifying that both models are working before using them in the Combined sequence.

- **Green text (top):** Whatever printed text the OCR model can read in the frame
- **Green text (below):** The face data: `[name, center_x, center_y, height]`

Press the **Stop** button or interrupt the kernel to exit.

In [18]:
# Ensure the correct models are loaded and the camera is open
# !!!This will NOT be needed on the actual competition. It is here for testing purposes only.
got.open_camera()

try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Overlay any detected text in the top-left corner
        name = got.get_words_result()
        if name:
            cv2.putText(img, f"text: {name}", (20, 40), 0, 0.8, (0, 255, 0), 2)

        # Overlay face recognition data if any faces are detected.
        # Each face entry is: [name, center_x, center_y, height]
        faces = got.get_face_recognition_total_info()
        if faces:
            face = faces[0]  # Only look at the first detected face
            cv2.putText(img, f"face: {face}", (20, 70), 0, 0.8, (0, 255, 0), 2)

        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()

No camera data received


TypeError: a bytes-like object is required, not 'NoneType'

## Helper Functions

This section defines three reusable building blocks that the main mission loop (below) relies on. Each function handles one distinct robot skill:

| Function | What it does |
|---|---|
| `AP_centralization_approaching()` | Drives toward a detected AprilTag while keeping it centered in frame |
| `pick_up()` | Uses the robot arm to grasp the object identified by the nearest AprilTag |
| `line_follow()` | Follows a line on the ground, steering proportionally to stay centered |

Define all three before running the **Combined** section.

### `AP_centralization_approaching()` — AprilTag Approach

This function drives the robot toward a visible AprilTag marker, continuously correcting its lateral position to keep the tag centered in the camera frame. It stops automatically once the tag is within the target distance.

The camera frame is 640 px wide, so **center = 320 px**. The robot strafes left or right whenever the tag drifts beyond `gap` pixels from center, and drives straight forward when alignment is good.

**Parameters:**
| Parameter | Default | Description |
|---|---|---|
| `distance` | `0.15` m | Stop when the tag is within this many meters |
| `gap` | `20` px | Pixel tolerance around center before triggering a strafe correction |
| `fwd_spd` | `10` | Forward drive speed (cm/s) |
| `strafe_spd` | `10` | Left/right correction speed (cm/s) |

**Control logic (per frame):**
```
tag x < 300  ->  strafe LEFT
tag x > 340  ->  strafe RIGHT
distance > threshold  ->  drive FORWARD
otherwise  ->  STOP and exit
```

### `pick_up()` — Arm Grasp

Once the robot is close and aligned (i.e. `AP_centralization_approaching()` has just returned), `pick_up()` calculates arm joint angles from the tag's current position and executes the grasp sequence.

**Arm joint mapping:**
| Joint | Role | Calculation |
|---|---|---|
| `joint1` | Base rotation (left/right) | `(AP_x - 320) × -1 / 10` — negative factor corrects for horizontal camera mirroring |
| `joint3` | Extension (reach distance) | `AP_distance × 100 - 80` — converts meters to degrees; `-80` accounts for resting angle |

**Grasp sequence:**
1. Move arm to neutral ready pose and open gripper
2. Calculate `joint1` and `joint3` from the live tag reading
3. Extend arm to the computed pick-up pose
4. Close gripper
5. Return arm to neutral carry pose

In [25]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """
    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
    AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

    while True:
        # Refresh tag data every iteration for responsive corrections.
        AP_info = got.get_apriltag_total_info()
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        if AP_x < 320 - gap:
            # Tag is to the LEFT of center — strafe left to re-align.
            # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
            got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
        elif AP_x > 320 + gap:
            # Tag is to the RIGHT of center — strafe right to re-align.
            got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
        elif AP_distance > distance:
            # Tag is centered but still too far — drive straight forward.
            got.mecanum_move_xyz(0, fwd_spd, 0)
        else:
            # Tag is centered AND within target distance — stop and exit.
            got.mecanum_stop()
            print("It's too close, let's stop.")
            break


def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    try:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # Move arm to a neutral ready position and open the gripper.
        # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts arm slightly forward.
        got.mechanical_joint_control(0, 30, 30, 1000)
        got.mechanical_clamp_release() # Open gripper before extending arm
        time.sleep(2) # Wait for gripper to fully open

        # Calculate arm joint angles based on the tag's camera position.
        # joint1 (base): convert pixel offset from center to degrees.
        #   Negative factor corrects for the camera being mirrored horizontally.
        joint1 = int((AP_x - 320) * -1 / 10)

        # joint3 (furthest): convert distance (m) to an extension angle.
        # The -80 offset accounts for the arm's resting angle calibration.
        joint3 = int(AP_distance * 100 - 80)

        # Move arm to the computed pick-up pose.
        got.mechanical_joint_control(joint1, 0, joint3, 500)
        print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
        time.sleep(1) # Wait for arm to reach the target pose

        # Grasp the object and lift the arm back to the carry position.
        got.mechanical_clamp_close()
        time.sleep(2)  # Wait for gripper to fully close before lifting
        got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose
    except IndexError:
        print("AprilTag is too close!")

### `face_find_and_approach` — Find a Person and Walk Up to Them

This function combines face recognition and movement to make the robot:
1. **Spin** in place until it spots the target person (by face or by reading their name tag)
2. **Approach** the person, keeping them centered in frame, until it gets close enough

**How the approach works:**
- The camera frame is 640 px wide, so the center is at x = 320.
- If the face center (`c_x`) is too far **left** of center, the robot strafes left while moving forward.
- If `c_x` is too far **right**, it strafes right while moving forward.
- If the face is roughly centered but the detected face **height** (`h`) is still small (meaning the robot is far away), it moves straight forward.
- Once the face height exceeds the `height` threshold (meaning the robot is close enough), it stops.
- If the face is lost mid-approach, the robot inches forward slowly until it reappears.

**Parameters:**
| Parameter | Default | Description |
|---|---|---|
| `gap` | `10` | Pixel tolerance around the center (x = 320 ± gap). Within this band the robot won't strafe. |
| `target_name` | `"Stranger"` | The name to search for; must match either the face recognition label or text the OCR can read. |
| `turn_spd` | `15` | Speed used when spinning to search for the target. |
| `strafe_spd` | `10` | Sideways speed used to re-center the target in frame during approach. |
| `fwd_spd` | `10` | Forward speed used during approach. |
| `height` | `80` | How tall (in pixels) the face bounding box must be before the robot considers itself close enough to stop. Decrease this to stop further away; increase it to get closer. |
| `adjust_turn` | `10` | How far the robot turns to center itself after first spotting the target, in degree-units. Increase if the robot tends to approach at an angle. |

In [26]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

            clear_output(wait=True)

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

### `line_follow()` — Line Following Helper

This small helper wraps the robot's line-tracking API into a single convenient call. Each time it's called it:
- Reads the current line detection from the robot
- Calculates a rotation speed proportional to how far off-center the line is
- Sends a combined forward + rotation movement command
- Returns the detection info so the caller can make navigation decisions

**Parameters:**
| Parameter | Default | Description |
|---|---|---|
| `mult` | `0.25` | Multiplier converting line offset into rotation speed. Higher = sharper corrections; lower = gentler steering. |
| `speed` | `35` | Forward speed while following the line. |

**Return values:** `(line_type, x, y)`
| Value | Description |
|---|---|
| `line_type` | `0` = no line detected, `1` = normal line, `2` = y-intersection (fork in the path) |
| `x`, `y` | Pixel coordinates of the detected line position |

In [27]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y

In [48]:
got.set_track_recognition_line(0)
got.open_camera()

try:
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=10)

        if line_type == 0:  # 0 = no line detected,
            print("No line!")
            break

        if line_type == 2:  # 2 = y-intersection detected
            print("Y-intersection!")
            break

    got.mecanum_stop()

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

No line!


## Combined — Color -> AprilTag Pickup -> Line Follow

This section chains all the robot's skills into a single end-to-end autonomous program. The robot works through **three sequential phases**:

| Phase | Trigger | What the robot does |
|---|---|---|
| **1 — Color sequence** | Camera color recognition | Waits to see Red, then Green, in order |
| **2 — AprilTag pickup** | AprilTag detection | Approaches and picks up the tagged object |
| **3 — Line follow to junction** | Line tracking | Follows the line until it ends or forks |

In [13]:
red_seen = False  # Flag to track if Red has been detected yet
got.set_track_recognition_line(line_type=0)  # 0 is a single track line

try:
    while True:
        color = got.get_color_total_info()[0]  # Read fresh color data each loop

        if not red_seen:
            if color == "Red":
                got.screen_display_background(3)  # Red background
                print("Red detected!")
                red_seen = True  # Unlock green detection
        else:
            if color == "Green":
                got.screen_display_background(6)  # Green background
                print("Green detected!")
                time.sleep(1)
                break  # Both colors seen — exit loop

    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            AP_centralization_approaching(
                distance=0.14, gap=18, fwd_spd=11, strafe_spd=5
            )
            pick_up()
            print("Picked up!")
            time.sleep(2)
            break
        else:
            got.mecanum_move_speed(0, 15)

    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=20)

        if line_type == 0:  # 0 = no line detected,
            print("No line!")
            break

        if line_type == 2:  # 2 = y-intersection detected
            print("Y-intersection!")
            break

    # If you want to go forward a bit to move closer to the text:
    got.mecanum_move_speed_times(direction=0, speed=20, times=10, unit=1)

    # or turn
    got.mecanum_turn_speed_times(turn=2, speed=30, times=15, unit=2)

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

Red detected!
Green detected!


IndexError: list index out of range

## Combined - Text Recognition -> Direction Decision

After the robot arrives at the end of the line, this final cell uses the `word_recognition` model to read a printed sign and decide which way to turn.

**How it works:**

1. **Read text from the camera** — `got.get_words_result()` queries the OCR model and returns any text currently visible in the frame as a string.
2. **Match a command word** — The result is compared against known direction commands:

| Text detected | Action |
|---|---|
| `"LEFT"` | `mecanum_turn_speed_times(turn=2, ...)` — counter-clockwise ~45° |
| `"RIGHT"` | `mecanum_turn_speed_times(turn=3, ...)` — clockwise ~45° |

3. **Execute the turn** - Once a matching word is read, the robot turns in that direction and `break`s out of the detection loop.
4. **Resume line following** — A second `while` loop calls `line_follow()` to drive forward along the next line segment, stopping when the line ends (`line_type == 0`).

> **Note:** The loop polls `get_words_result()` continuously until a recognized command appears. If the sign is not yet in frame or the OCR confidence is low, it will simply keep trying. Position the robot so the sign is clearly visible before running this cell.

In [1]:
try:
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        # React to specific command words
            # Turn counter-clockwise by ~45 degrees
        if text in ["LEFT", "RIGHT"]:
            turn = 2
            if(text == "RIGHT"): turn = 3
            while True:
                got.mecanum_turn_speed_times(turn=turn, speed=30, times=20, unit=2)
                line_type, _, _ = line_follow(mult=0, speed=0)
                if line_type == 1:
                    print("Turned and see line")
                    break
            break

    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=15)
        if line_type in [0,2]:
            break   

    got.mecanum_stop() 

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

NameError: name 'got' is not defined

## Combining EVERYTHING

In [ ]:
red_seen = False  # Flag to track if Red has been detected yet
got.set_track_recognition_line(line_type=0)  # 0 is a single track line
TARGET_NAME = "Ryan"

try:
    # Color Recognition
    print("Starting color recognition...")
    while True:
        color = got.get_color_total_info()[0]  # Read fresh color data each loop

        if not red_seen:
            if color == "Red":
                got.screen_display_background(3)  # Red background
                print("Red detected!")
                red_seen = True  # Unlock green detection
        else:
            if color == "Green":
                got.screen_display_background(6)  # Green background
                print("Green detected!")
                time.sleep(1)
                break  # Both colors seen — exit loop

    # AprilTag Recognition
    print("Starting AprilTag recognition...")
    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            AP_centralization_approaching(
                distance=0.13, gap=20, fwd_spd=10, strafe_spd=5
            )
            pick_up()
            print("Picked up!")
            time.sleep(2)
            break
        else:
            got.mecanum_move_speed(0, 15)

    # Line follow pt 1
    print("Starting line following...")
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=20)

        if line_type == 0:  # 0 = no line detected,
            print("No line!")
            break

        if line_type == 2:  # 2 = y-intersection detected
            print("Y-intersection!")
            break

    got.mecanum_stop()

    # If you want to go forward a bit to move closer to the text:
    got.mecanum_move_speed_times(direction=0, speed=20, times=20, unit=1)

    # or turn
    got.mecanum_turn_speed_times(turn=2, speed=30, times=15, unit=2)

    # Text Recognition
    print("Starting text recognition...")
    while True:
        text = got.get_words_result()

        print("Text:", text)

        # React to specific command words
        if text == "LEFT":
            # Turn counter-clockwise by ~45 degrees
            got.mecanum_turn_speed_times(turn=2, speed=20, times=45, unit=2)
            break
        elif text == "RIGHT":
            # Turn clockwise by ~45 degrees
            got.mecanum_turn_speed_times(turn=3, speed=20, times=45, unit=2)
            break

    # Line follow pt 2
    print("Starting line follow...")
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=35)

        if line_type in [0,2]:
            break

    # Pose recognition
    print("Starting pose recognition...")
    run_pose_control_inline(
        robot_ip="192.168.88.1",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got,
    )
    time.sleep(2)
    clear_output()

    # Face recognition
    print("Starting face recognition...")
    face_find_and_approach(
        gap=15,
        target_name=TARGET_NAME,
        turn_spd=10,
        strafe_spd=10,
        fwd_spd=10,
        height=120,
        adjust_turn=10,
    )
    got.mecanum_move_speed_times(
        direction=1, speed=20, times=15, unit=1
    )  # Go backwards a bit to not hit face
    got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-30, duration=500)
    time.sleep(1)
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

Starting color recognition...
Red detected!
Green detected!
Starting AprilTag recognition...
Done!


In [32]:
try:
    print("Starting pose recognition...")
    run_pose_control_inline(
        robot_ip="192.168.88.1",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got,
    )
    time.sleep(2)
    clear_output()

    # Face recognition
    print("Starting face recognition...")
    face_find_and_approach(
        gap=15,
        target_name=TARGET_NAME,
        turn_spd=10,
        strafe_spd=10,
        fwd_spd=10,
        height=120,
        adjust_turn=10,
    )
    got.mecanum_move_speed_times(
        direction=1, speed=20, times=15, unit=1
    )  # Go backwards a bit to not hit face
    got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-30, duration=500)
    time.sleep(1)
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)
except KeyboardInterrupt:
    print("Done")

Starting face recognition...
Done


In [ ]:
got.mechanical_clamp_release()